# African Biosecurity LLM Evaluation

**Evaluating how well open-source models understand Nigerian livestock management and ethnoveterinary knowledge**

This project tests model performance on local African context, traditional practices, and livestock knowledge.

In [ ]:
# Install dependencies
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "openai", "pandas", "tqdm", "-q"])

In [ ]:
import os
import json
import time
import re
import glob
from datetime import datetime
from getpass import getpass

import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI

In [ ]:
# ====================== CONFIGURATION ======================
API_KEY = getpass("Enter your Groq API key (gsk_...): ")

MODELS_TO_EVALUATE = [
    "llama-3.1-8b-instant",
    "gemma2-9b-it",
]

JUDGE_MODEL = "llama-3.1-8b-instant"
CHECKPOINT_DIR = "checkpoints"
CHECKPOINT_EVERY = 70
CALL_DELAY = 10.0
MAX_RETRIES = 5

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Setup complete - {len(MODELS_TO_EVALUATE)} models to evaluate")

In [ ]:
# Groq client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=API_KEY
)

In [ ]:
def call_model(model, prompt, max_retries=MAX_RETRIES):
    """Call Groq model with retry logic."""
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=512,
                timeout=60
            )
            content = resp.choices[0].message.content
            return content.strip() if content else None
        
        except Exception as e:
            err = str(e).lower()
            if "not found" in err or "404" in err:
                print(f"Model unavailable: {model}")
                return None
            
            wait = 30
            match = re.search(r"try again in (\d+\.?\d*)s", err)
            if match:
                wait = float(match.group(1)) + 5
            
            time.sleep(wait)
    return None

In [ ]:
def extract_json(text):
    if not text:
        return None
    try:
        return json.loads(text.strip())
    except:
        pass
    cleaned = re.sub(r"```(?:json)?", "", text, flags=re.IGNORECASE).strip().strip("`")
    try:
        return json.loads(cleaned)
    except:
        pass
    return None


JUDGE_PROMPT = """You are an expert judge evaluating AI responses on Nigerian livestock management and ethnoveterinary practices.

QUESTION: {question}
REFERENCE: {correct_answer}
MODEL ANSWER: {model_response}

Score 0-2:
0 = Wrong or ignores Nigerian/African local context
1 = Partially correct
2 = Fully correct and contextually accurate

Return only JSON: {{"score": <0|1|2>, "reasoning": "short explanation"}}"""

def judge_response(question, correct_answer, model_response):
    if not model_response:
        return {"score": -1, "reasoning": "No response from model"}
    
    prompt = JUDGE_PROMPT.format(
        question=question,
        correct_answer=correct_answer,
        model_response=model_response
    )
    
    raw = call_model(JUDGE_MODEL, prompt, max_retries=3)
    parsed = extract_json(raw)
    
    if parsed and isinstance(parsed.get("score"), (int, float)):
        return {"score": int(parsed["score"]), "reasoning": str(parsed.get("reasoning", ""))[:250]}
    return {"score": -1, "reasoning": "Judging failed"}

In [ ]:
# Load dataset
csv_files = glob.glob("*.csv")
if csv_files:
    df = pd.read_csv(csv_files[0])
else:
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(list(uploaded.keys())[0])

print(f"Loaded {len(df)} questions")
print("\nCategories:")
print(df["Category"].value_counts())

In [ ]:
# TODO: Add your pilot test + full evaluation code here
# You can run small tests first before running on the full dataset